# Sensitivity Analysis and Robustness Checks

*Companion notebook — Quasi-Experimental Methods for Business Interventions in Python. Self-contained: run the setup cells first.*

In [ ]:
# Install dependencies (Colab/Jupyter)
%pip install -q numpy pandas matplotlib seaborn statsmodels linearmodels scikit-learn scipy doubleml econml dowhy tfp-causalimpact pysyncon rdrobust

## Setup: shared imports and data generators

This cell reproduces the book's `_common.py` so the notebook runs standalone.

In [ ]:
"""Shared setup for all chapters: imports, plotting theme, and data generators."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


def make_panel(
    n_stores=120,
    n_markets=12,
    n_weeks=104,                # 2 years of weekly data
    treat_markets=(0, 1, 2, 3), # which markets get the program
    treat_week=60,              # program launch week
    true_effect=8.0,            # true lift in weekly revenue ($k)
    effect_ramp=6,              # weeks for the effect to reach full size
    seed=42,
):
    """NorthShore Retail store-week panel with known treatment effect."""
    rng = np.random.default_rng(seed)
    stores = np.arange(n_stores)
    markets = stores % n_markets
    store_fe = rng.normal(100, 15, n_stores)
    market_fe = rng.normal(0, 8, n_markets)
    store_trend = rng.normal(0.05, 0.03, n_stores)

    rows = []
    market_shock = rng.normal(0, 3, (n_markets, n_weeks))
    for w in range(n_weeks):
        season = 6 * np.sin(2 * np.pi * w / 52)
        for s in stores:
            m = markets[s]
            treated_unit = m in treat_markets
            post = w >= treat_week
            if treated_unit and post:
                ramp = min(1.0, (w - treat_week + 1) / effect_ramp)
                tau = true_effect * ramp
            else:
                tau = 0.0
            y = (store_fe[s] + market_fe[m] + store_trend[s] * w
                 + season + market_shock[m, w]
                 + tau + rng.normal(0, 4))
            rows.append((s, m, w, int(treated_unit), int(post),
                         int(treated_unit and post), y))

    return pd.DataFrame(rows, columns=[
        "store", "market", "week", "treated", "post", "D", "revenue"])


def make_customers(n=8000, true_att=12.0, seed=42):
    """Loyalty-program adoption data with selection on observables."""
    rng = np.random.default_rng(seed)
    tenure = rng.gamma(3, 1.2, n)
    spend0 = rng.lognormal(4.0, 0.5, n)
    engage = rng.beta(2, 5, n) * 10
    urban = rng.binomial(1, 0.55, n)
    logit = -3.0 + 0.25*tenure + 0.004*spend0 + 0.30*engage + 0.4*urban
    p = 1 / (1 + np.exp(-logit))
    D = rng.binomial(1, p)
    y = (0.55*spend0 + 4*tenure + 2.5*engage + 6*urban
         + true_att * D + rng.normal(0, 12, n))
    return pd.DataFrame(dict(tenure=tenure, spend0=spend0, engage=engage,
                             urban=urban, D=D, y=y))


panel = make_panel()
cust = make_customers()
X_cols = ["tenure", "spend0", "engage", "urban"]
TRUE_ATT = 12.0


Identifying assumptions are unverifiable, so a credible evaluation quantifies **how wrong the assumption would have to be to change the conclusion**.

## The robustness battery (use as a checklist)

| Check | Targets | Implemented in |
|---|---|---|
| Pre-trend event-study plot | DiD parallel trends | Section 5.1 |
| Placebo-in-time (fake launch date) | ITS, CausalImpact, SCM, DiD | Sections 6–8 |
| Placebo-in-space (fake treated unit) | SCM, CausalImpact, DiD | Sections 7–8 |
| Placebo *outcome* (a metric the program can't affect, e.g., revenue in an untouched category) | All designs | trivial rerun |
| Density/manipulation test, covariate continuity | RDD | Section 10.4 |
| First-stage F, Anderson–Rubin CI | IV | Section 11.3 |
| Balance (SMD), overlap plots, weight distribution | PS methods, DML | Section 9 |
| Leave-one-out (donors, markets, time windows) | SCM, DiD | Section 8.3 |
| Specification curve (estimates across reasonable spec choices) | All | loop + plot |

## Quantifying unmeasured confounding: Rosenbaum-style and E-values

**E-value** (VanderWeele & Ding 2017): the minimum strength of association an unmeasured confounder would need with *both* treatment and outcome to fully explain away the estimate. Easy to compute and explain to stakeholders:

In [ ]:
def e_value_rr(rr):
    rr = max(rr, 1/rr)
    return rr + np.sqrt(rr * (rr - 1))

# convert an effect to an approximate risk ratio first (or use it for binary outcomes)
print(f"E-value for RR=1.4: {e_value_rr(1.4):.2f}")
# → a confounder must be associated with both D and Y at RR ≥ 2.15 to nullify

**Omitted-variable bias bounds for DML/regression** — `sensemakr`-style analysis (Cinelli & Hazlett 2020) asks "how strong must a confounder be, *relative to an observed benchmark covariate*, to overturn the result?" The `DoubleML` package ships this directly:

In [ ]:
plr.sensitivity_analysis(cf_y=0.03, cf_d=0.03)   # hypothetical confounding strength
print(plr.sensitivity_summary)
plr.sensitivity_plot()                           # contour of bias vs. confounder strength

The deliverable sentence stakeholders need: *"To explain away the $12 effect, an unmeasured confounder would have to be more strongly related to adoption and spend than baseline spend itself — implausible given what we measure."*

## Honest reporting

- Report the **design and assumption**, not just the number ("DiD assuming parallel trends, supported by flat pre-trends over 59 weeks").
- Report the **estimand** (ATT for treated markets; LATE for email compliers).
- Show **uncertainty appropriate to the design** (clustered/wild-bootstrap SEs, permutation p-values, posterior intervals).
- Pre-specify primary outcome, model, and donor pools before looking at post-period data when possible — the observational analogue of a pre-registered A/B test. A one-page "analysis plan" doc beats a post-hoc garden of forking paths.